# Yahoo Finance Data Extraction & DVC Tracking

This notebook provides a utility function to download stock/financial data from Yahoo Finance (`yfinance`) and automatically track and upload it to the project's Oracle Cloud DVC remote.

**Prerequisite:** Make sure you have successfully authenticated with Oracle via DVC (run `00-colab-setup.ipynb` if you are in Colab).

In [ ]:
import yfinance as yf
import pandas as pd
import subprocess
import os

In [ ]:
def extract_and_store_yahoo_data(name: str, folder_type: str = 'raw', period: str = '1y'):
    """
    Extracts data from Yahoo Finance, saves it to the specified data folder, 
    and tracks/pushes it via DVC to the Oracle bucket.
    
    Parameters:
    - name (str): The Yahoo Finance ticker symbol (e.g., 'AAPL'). Will be used as the filename.
    - folder_type (str): The subfolder in the data directory (e.g., 'raw', 'interim').
    - period (str): The time period to fetch from Yahoo Finance (e.g., '1mo', '1y', 'max').
    """
    # Define the output directory based on project structure 
    # Assuming the notebook is run from the 'notebooks' folder
    base_dir = f"../data/{folder_type}"
    os.makedirs(base_dir, exist_ok=True)
    file_path = f"{base_dir}/{name}.csv"
    
    print(f"📉 Fetching '{period}' data for {name}...")
    ticker = yf.Ticker(name)
    df = ticker.history(period=period)
    
    if df.empty:
        print(f"❌ No data found for {name}. Please check the ticker name.")
        return
        
    # Save to CSV
    df.to_csv(file_path)
    print(f"💾 Data saved locally to {file_path}")
    
    # Track with DVC
    print("📦 Tracking file with DVC...")
    subprocess.run(["dvc", "add", file_path], check=True)
    
    # Push to Oracle Cloud Object Storage
    print("☁️ Pushing to Oracle Cloud Bucket...")
    # DVC outputs the file name with a .dvc extension
    dvc_file = f"{file_path}.dvc"
    subprocess.run(["dvc", "push", dvc_file], check=True)
    
    print(f"✅ Success! {name} data is now version-controlled and stored in the Oracle bucket.")

In [ ]:
# Example Usage:
# This will download 1 month of Apple stock data, save it to ../data/raw/AAPL.csv, 
# track it with DVC, and upload it to the Oracle bucket.

# extract_and_store_yahoo_data(name='AAPL', folder_type='raw', period='1mo')